# Cross-Africa Satellite Prediction: Augmenting Labels (NBK 2)

This notebook enriches the labels table with two columns needed by the downstream analyses:
- `region_pop` (used for weighting the policy counterfactual analysis in the planned Nigeria case study),
- `geo_region` (West/East/Central/Southern/North Africa: used as an explanatory variable in the country-level R² regression).

The output, `labels_augmented.csv`, is the canonical labels file for notebooks 2–6.

---

### Import statements

In [1]:
import os
import pandas as pd
import helpers

---

### Loading the existing labels and GDL SHDI population column

The SHDI file carries a `pop` column (regional population, in thousands) for every ADM1 region across years. The code picks each region's `pop` observation closest to 2019, matching the same year-selection logic used for the other outcomes during dataset construction.

In [2]:
labels = pd.read_csv(os.path.join(helpers.PROC, "labels.csv"))
shdi = pd.read_csv(os.path.join(helpers.GDL_DIR, "Subnational HDI Data v10.2.csv"))
shdi.columns = shdi.columns.str.lower()
shdi = shdi[(shdi["level"] == "Subnat") & (shdi["continent"] == "Africa")]
shdi["gap"] = (shdi["year"] - 2019).abs()
shdi = shdi.sort_values(["gdlcode","gap","year"])
pop = shdi.drop_duplicates(subset="gdlcode", keep="first")[["gdlcode","pop"]]
pop.columns = ["gdlcode","region_pop"]

labels = labels.merge(pop, on="gdlcode", how="left")
print(f"Population coverage: {labels['region_pop'].notna().sum()} / {len(labels)} regions")

Population coverage: 544 / 564 regions


Regional population will be used in the planned Nigeria policy counterfactual to convert "regions missed" into "people missed", which is a more interpretable policy metric.

---

### Assigning each country to a geographic region of Africa

The African country-to-region mapping is hand-coded in `helpers.AFRICA_REGION` (West/East/Central/Southern/North). This classification is used downstream as an explanatory variable in the country-level R² regression, the hypothesis being that geographic proximity in training data correlates with prediction accuracy.

In [6]:
labels["geo_region"] = labels["isocode3"].map(helpers.AFRICA_REGION)
missing = labels[labels["geo_region"].isna()]["isocode3"].unique()
if len(missing):
    print("WARNING unmapped countries:", missing)

print("Countries per geographic region:\n")
print(labels.groupby("geo_region")["isocode3"].nunique())

Countries per geographic region:

geo_region
Central      9
East        15
North        5
Southern     9
West        16
Name: isocode3, dtype: int64


The 54 African countries in the dataset split into 16 West / 15 East / 9 Central / 9 Southern / 5 North. As a methodological choice, Sudan (`SDN`) is intentionally grouped with East Africa rather than North Africa based on contemporary political linkage.

---

### Saving the augmented labels

The enriched table is written to `data/processed/labels_augmented.csv` and serves as the labels input for downstream analysis.

In [7]:
out_path = os.path.join(helpers.PROC, "labels_augmented.csv")
labels.to_csv(out_path, index=False)
print(f"Saved: ({len(labels)} rows, {len(labels.columns)} cols)")

Saved: (564 rows, 34 cols)


It carries every outcome plus the two new metadata columns (`region_pop` and `geo_region`).

---